# How the data is transformed

The data passes through three phases before a model ever sees it. This notebook
shows what it looks like at each one, using real rows rather than descriptions.

| Phase | Code | What happens |
|---|---|---|
| 0. Raw | `data.load_raw` | the CSV exactly as provided |
| 1. Cleaned | `data.clean` | sign errors fixed, gaps filled, bad labels flagged |
| 2. Featured | `features.add_features` | engineered columns added |
| 3. Matrix | `features.feature_matrix` | the fixed set of model inputs |

No modelling and no validation here — those live in `04_evaluation.ipynb` and
`05_candidate_models.ipynb`. This is only about the shape of the data.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import clean, load_raw
from src.features import (
    FEATURE_COLUMNS,
    add_features,
    attach_coordinates,
    attach_market_signals,
    build_city_coordinates,
    daily_market_signals,
    feature_matrix,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## Phase 0 — raw

Straight from `data/train_test.csv`, untouched.

In [2]:
raw = load_raw("train_test.csv")
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head(3)

48,000 rows x 14 columns


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54


In [3]:
pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "nulls": raw.isna().sum(),
    "example": raw.iloc[0],
})

,dtype,nulls,example
load_id,object,0,TR-000001
pickup,object,0,Richmond
delivery,object,0,Baltimore
pickup_lat,float64,0,38.09122
pickup_lon,float64,0,-76.78906
delivery_lat,float64,0,38.16908
delivery_lon,float64,0,-72.74564
distance,float64,0,274.3
equipment,object,0,Dry Van
weight,float64,300,30658.0


Fourteen columns: an id, two city names, four coordinates, distance, equipment,
weight, date, two market signals, and the target `posted_rate`.

Two columns carry nulls (`weight`, `market_index`), and `weight` has a further
problem the null count does not reveal.

## Phase 1 — cleaned

`clean()` does four things. Three of them change values; the fourth only adds a
flag and never touches the target.

In [4]:
cleaned, report = clean(raw)
report.to_frame()

,count
rows,48000
weight_sign_fixed,292
weight_imputed,300
market_index_imputed,374
market_index_from_date,374
corrupted_flagged,677


### The same rows, before and after

Rather than trust the counts, here are actual rows that changed.

**Negative weights.** These are sign-entry errors: the magnitudes sit inside the
valid 5,000-47,500 lb range, so only the sign was lost.

In [5]:
sign_fixed_rows = raw.index[raw.weight < 0][:4]
pd.DataFrame({
    "before": raw.loc[sign_fixed_rows, "weight"],
    "after": cleaned.loc[sign_fixed_rows, "weight"],
})

,before,after
68,-36559.0,36559.0
204,-26670.0,26670.0
206,-20003.0,20003.0
225,-23981.0,23981.0


**Missing `market_index`.** Filled from the mean of other loads on the same
date, because `market_index` is a daily market level with very little
within-day spread.

In [6]:
imputed_rows = raw.index[raw.market_index.isna()][:4]
pd.DataFrame({
    "date": raw.loc[imputed_rows, "date"],
    "before": raw.loc[imputed_rows, "market_index"],
    "after": cleaned.loc[imputed_rows, "market_index"].round(5),
})

,date,before,after
116,2025-01-01,NaN,0.96122
226,2025-01-02,NaN,0.98661
311,2025-01-02,NaN,0.98661
511,2025-01-04,NaN,0.85770


**Corrupted labels.** Two new columns appear. `rate_ratio` is the load's price
per mile divided by the median for its equipment and distance peer group;
`is_corrupted` is `True` when that ratio falls outside 0.5x-2.0x.

Note that `posted_rate` itself is unchanged — these rows are marked, not edited
or removed. Excluding them is the model's decision, not the cleaner's.

In [7]:
flagged = cleaned.index[cleaned.is_corrupted][:4]
comparison = pd.DataFrame({
    "distance": cleaned.loc[flagged, "distance"],
    "equipment": cleaned.loc[flagged, "equipment"],
    "posted_rate_raw": raw.loc[flagged, "posted_rate"],
    "posted_rate_cleaned": cleaned.loc[flagged, "posted_rate"],
    "rate_ratio": cleaned.loc[flagged, "rate_ratio"].round(3),
    "is_corrupted": cleaned.loc[flagged, "is_corrupted"],
})
print("posted_rate identical before and after:",
      bool((comparison.posted_rate_raw == comparison.posted_rate_cleaned).all()))
comparison

posted_rate identical before and after: True


,distance,equipment,posted_rate_raw,posted_rate_cleaned,rate_ratio,is_corrupted
50,2202.5,Reefer,1243.45,1243.45,0.264,True
190,919.8,Dry Van,704.89,704.89,0.364,True
294,685.6,Dry Van,518.31,518.31,0.359,True
388,823.0,Flatbed,5485.49,5485.49,2.923,True


In [8]:
print("columns added by cleaning:", sorted(set(cleaned.columns) - set(raw.columns)))
print(f"rows: {len(raw):,} -> {len(cleaned):,}   (nothing dropped)")
print("remaining nulls:", int(cleaned[['weight', 'market_index', 'quote_signal']].isna().sum().sum()))

columns added by cleaning: ['is_corrupted', 'rate_ratio']
rows: 48,000 -> 48,000   (nothing dropped)
remaining nulls: 0


## Phase 2 — featured

`add_features()` derives new columns from the ones already present. Nothing is
removed and no external data is used.

In [9]:
featured = add_features(cleaned)
new_columns = [c for c in featured.columns if c not in cleaned.columns]
print(f"{len(new_columns)} columns added:")
for column in new_columns:
    print(f"  {column}")

13 columns added:
  log_distance
  weight_per_mile
  haversine_distance
  circuity
  bearing_sin
  bearing_cos
  days_since_origin
  dow_sin
  dow_cos
  is_weekend
  equipment_dry_van
  equipment_flatbed
  equipment_reefer


### What those columns hold, for one real load

In [10]:
row = featured.iloc[0]

groups = {
    "load": ["distance", "log_distance", "weight", "weight_per_mile"],
    "geography": ["pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon",
                  "haversine_distance", "circuity", "bearing_sin", "bearing_cos"],
    "market": ["market_index", "quote_signal"],
    "time": ["days_since_origin", "dow_sin", "dow_cos", "is_weekend"],
    "equipment": [c for c in FEATURE_COLUMNS if c.startswith("equipment_")],
}

print(f"load {row.load_id}:  {row.pickup} -> {row.delivery}, {row.equipment}, "
      f"{row.date.date()} ({row.date.day_name()})\n")
pd.concat(
    [pd.Series({c: row[c] for c in columns}, name="value").to_frame().assign(group=group)
     for group, columns in groups.items()]
)[["group", "value"]]

load TR-000001:  Richmond -> Baltimore, Dry Van, 2025-01-01 (Wednesday)



,group,value
distance,load,274.300000
log_distance,load,5.614222
weight,load,30658.000000
weight_per_mile,load,111.768137
pickup_lat,geography,38.091220
pickup_lon,geography,-76.789060
delivery_lat,geography,38.169080
delivery_lon,geography,-72.745640
haversine_distance,geography,219.806619
circuity,geography,1.247915


Reading a few of these:

* `log_distance` — the log of distance, because rate is close to proportional to
  distance and that relationship is linear in logs.
* `haversine_distance` — straight-line miles between the two coordinate pairs;
  `circuity` is reported distance divided by it, normally around 1.18.
* `bearing_sin` / `bearing_cos` — direction of travel on a circle, so that 359
  degrees and 1 degree sit next to each other instead of at opposite extremes.
* `days_since_origin` — days since 2025-01-01. A single continuous number, which
  is what lets a linear model carry the price trend past the training window.
* `dow_sin` / `dow_cos` — day of week on a circle, so Sunday and Monday are
  adjacent. Kept separate from the trend because a week repeats and a trend
  does not.
* `equipment_*` — one column per type, holding 1 or 0.

## Phase 3 — the feature matrix

`feature_matrix()` selects `FEATURE_COLUMNS` in a fixed order. Everything the
model does not consume is dropped here: ids, city names, the raw equipment
string, the date, and the target.

In [11]:
matrix = feature_matrix(featured)
print(f"{matrix.shape[0]:,} rows x {matrix.shape[1]} columns")
print("\ndropped at this step:", sorted(set(featured.columns) - set(matrix.columns)))
matrix.head(3)

48,000 rows x 21 columns

dropped at this step: ['date', 'delivery', 'equipment', 'is_corrupted', 'load_id', 'pickup', 'posted_rate', 'rate_ratio']


,distance,log_distance,weight,weight_per_mile,pickup_lat,pickup_lon,delivery_lat,delivery_lon,haversine_distance,circuity,bearing_sin,bearing_cos,market_index,quote_signal,days_since_origin,dow_sin,dow_cos,is_weekend,equipment_dry_van,equipment_flatbed,equipment_reefer
0,274.3,5.614222,30658.0,111.768137,38.09122,-76.78906,38.16908,-72.74564,219.806619,1.247915,0.998930,0.046241,0.95684,2.39595,0,0.974928,-0.222521,0,1,0,0
1,280.5,5.636574,17555.0,62.584670,38.09122,-76.78906,39.22317,-72.96710,220.523461,1.271974,0.927466,0.373908,0.97623,2.43355,0,0.974928,-0.222521,0,0,0,1
2,967.8,6.875025,31721.0,32.776400,39.22317,-72.96710,44.30296,-87.52871,826.911763,1.170379,-0.867699,0.497090,1.00971,1.84491,0,0.974928,-0.222521,0,1,0,0


In [12]:
pd.DataFrame({
    "phase": ["0. raw", "1. cleaned", "2. featured", "3. matrix"],
    "rows": [len(raw), len(cleaned), len(featured), len(matrix)],
    "columns": [raw.shape[1], cleaned.shape[1], featured.shape[1], matrix.shape[1]],
}).set_index("phase")

,rows,columns
phase,,
0. raw,48000,14
1. cleaned,48000,16
2. featured,48000,29
3. matrix,48000,21


Row count never changes — no phase drops a load. The column count grows through
cleaning and feature engineering, then narrows to the fixed set the model reads.

## The same pipeline on the December chart inputs

`december_chart_inputs.csv` is the awkward one: it ships with only seven columns
and is missing both market signals and all four coordinates. Two extra steps
rebuild them from data we were already given, and then it runs through the
identical phases.

In [13]:
december_raw = load_raw("december_chart_inputs.csv")
validation, _ = clean(load_raw("validation.csv"))
print(f"as provided: {december_raw.shape[0]} rows x {december_raw.shape[1]} columns")
print("missing vs validation:",
      sorted(set(validation.columns) - set(december_raw.columns) - {"load_id"}))
december_raw.head(3)

as provided: 31 rows x 7 columns
missing vs validation: ['delivery_lat', 'delivery_lon', 'market_index', 'pickup_lat', 'pickup_lon', 'quote_signal']


,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,NaN
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,NaN
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,NaN


In [14]:
# Coordinates from the city lookup; market signals from the December rows of
# validation.csv, which cover all 31 days. Neither uses the target.
city_coordinates = build_city_coordinates(cleaned, validation)
december_signals = daily_market_signals(validation[validation.date.dt.month == 12])

december = attach_coordinates(december_raw, city_coordinates)
december = attach_market_signals(december, december_signals)
december["weight"] = december.weight.abs()

december[["date", "pickup", "pickup_lat", "pickup_lon",
          "delivery_lat", "delivery_lon", "market_index", "quote_signal"]].head(3).round(4)

,date,pickup,pickup_lat,pickup_lon,delivery_lat,delivery_lon,market_index,quote_signal
0,2025-12-01,Lexington,36.9915,-84.9988,41.3156,-85.3621,0.8349,2.0520
1,2025-12-02,Lexington,36.9915,-84.9988,41.3156,-85.3621,0.9147,2.0341
2,2025-12-03,Lexington,36.9915,-84.9988,41.3156,-85.3621,0.9941,2.0533


In [15]:
december_matrix = feature_matrix(add_features(december))
print(f"december matrix: {december_matrix.shape[0]} rows x {december_matrix.shape[1]} columns")
print(f"nulls: {int(december_matrix.isna().sum().sum())}")
print(f"identical columns and order to the training matrix: "
      f"{list(december_matrix.columns) == list(matrix.columns)}")

december matrix: 31 rows x 21 columns
nulls: 0
identical columns and order to the training matrix: True


The December file ends up with exactly the same 21 columns in the same order as
the training matrix, with no nulls — so the fitted model can score it without
any special handling.

One detail worth noticing: every December row has identical load features by
design, and only `date` varies. So the only columns that differ between those 31
rows are the time and market ones.

In [16]:
varying = [c for c in december_matrix.columns if december_matrix[c].nunique() > 1]
print("columns that vary across the 31 December rows:")
for column in varying:
    print(f"  {column}")
december_matrix[varying].head(5).round(4)

columns that vary across the 31 December rows:
  market_index
  quote_signal
  days_since_origin
  dow_sin
  dow_cos
  is_weekend


,market_index,quote_signal,days_since_origin,dow_sin,dow_cos,is_weekend
0,0.8349,2.0520,334,0.0000,1.0000,0
1,0.9147,2.0341,335,0.7818,0.6235,0
2,0.9941,2.0533,336,0.9749,-0.2225,0
3,1.0240,2.0657,337,0.4339,-0.9010,0
4,0.9751,2.0474,338,-0.4339,-0.9010,0


## Summary

* **Phase 0 → 1:** in this training file, 292 sign-flipped weights corrected,
  300 weights and 374 market indices imputed, 677 labels flagged. No rows
  dropped, no target values changed. (`validation.csv` goes through the same
  code and needs it about twice as often per row.)
* **Phase 1 → 2:** 13 engineered columns added, all derived from columns already
  present.
* **Phase 2 → 3:** narrowed to the 21 columns in `FEATURE_COLUMNS`, dropping
  identifiers, the raw strings, the date and the target.
* The December chart inputs reach the same 21 columns in the same order after
  their missing pieces are rebuilt from the city lookup and validation.csv.